# Step 16b: Financial Attrition Cost Exposure Calculator

## Overview & Methodology Note
This notebook calculates estimated financial cost exposure resulting from predicted employee turnover:
- **Cost Formula**: $\text{Expected Cost} = (\text{MonthlyIncome} \times 12) \times \text{Cost Multiplier} \times \text{Attrition Probability}$
- **Configurable Multiplier Note**: The default cost multiplier is set to **1.5x annual salary** (industry standard replacement cost including recruiting, onboarding, and lost productivity). This assumption is fully configurable in `app/utils/config.py`.


In [1]:
import pandas as pd
import numpy as np
import os

PROCESSED_DIR = os.path.join("..", "data", "processed")
master_df = pd.read_csv(os.path.join(PROCESSED_DIR, "employee_intelligence_master.csv"))

COST_MULTIPLIER = 1.5 # Default 150% of annual salary

print(f"Loaded master employee intelligence dataset: {len(master_df)} rows")


Loaded master employee intelligence dataset: 1470 rows


---
## 1. Calculate Individual & Aggregated Financial Cost Exposure


In [2]:
master_df['Annual_Salary'] = master_df['MonthlyIncome'] * 12
master_df['Turnover_Replacement_Cost'] = master_df['Annual_Salary'] * COST_MULTIPLIER
master_df['Expected_Cost_Exposure'] = np.round(master_df['Turnover_Replacement_Cost'] * master_df['Attrition_Probability'], 2)

total_exposure = master_df['Expected_Cost_Exposure'].sum()
high_risk_exposure = master_df[master_df['Attrition_Risk_Tier']=='HIGH']['Expected_Cost_Exposure'].sum()

print(f"💰 TOTAL ORGANIZATION-WIDE COST EXPOSURE: ${total_exposure:,.2f}")
print(f"🔥 HIGH-RISK EMPLOYEES COST EXPOSURE    : ${high_risk_exposure:,.2f}")

print("\n=== Cost Exposure by Department ===")
dept_cost = master_df.groupby('Department').agg(
    Employee_Count=('EmployeeNumber', 'count'),
    High_Risk_Count=('Attrition_Risk_Tier', lambda x: (x == 'HIGH').sum()),
    Total_Cost_Exposure=('Expected_Cost_Exposure', 'sum'),
    Avg_Cost_Per_Emp=('Expected_Cost_Exposure', 'mean')
).reset_index().sort_values(by='Total_Cost_Exposure', ascending=False)
print(dept_cost.to_string(index=False))

print("\n=== Cost Exposure by Risk Tier ===")
tier_cost = master_df.groupby('Attrition_Risk_Tier').agg(
    Employee_Count=('EmployeeNumber', 'count'),
    Total_Cost_Exposure=('Expected_Cost_Exposure', 'sum')
).reset_index()
print(tier_cost.to_string(index=False))

out_path = os.path.join(PROCESSED_DIR, "attrition_cost_exposure_summary.csv")
dept_cost.to_csv(out_path, index=False)
print(f"\nSaved Cost Exposure Summary: {out_path}")


💰 TOTAL ORGANIZATION-WIDE COST EXPOSURE: $49,776,940.21
🔥 HIGH-RISK EMPLOYEES COST EXPOSURE    : $27,560,102.53

=== Cost Exposure by Department ===
            Department  Employee_Count  High_Risk_Count  Total_Cost_Exposure  Avg_Cost_Per_Emp
Research & Development             961              228          25034872.18      26050.855546
                 Sales             446              191          21891993.93      49085.188184
       Human Resources              63               20           2850074.10      45239.271429

=== Cost Exposure by Risk Tier ===
Attrition_Risk_Tier  Employee_Count  Total_Cost_Exposure
               HIGH             439          27560102.53
                LOW             770          11074843.14
             MEDIUM             261          11141994.54

Saved Cost Exposure Summary: ..\data\processed\attrition_cost_exposure_summary.csv
